# GDC Cleaning - RNA-Seq Counts và Metadata Mẫu

Notebook này chuẩn hóa dữ liệu GDC TCGA-LUAD từ HDFS raw sang HDFS refined ở dạng Parquet.

Input chính:
- `hdfs://master11:9000/drugtarget/data/raw/gdc/counts/file_id=*/*.tsv`
- `hdfs://master11:9000/drugtarget/data/raw/gdc/metadata/files_index/run_date=*/files_index_from_*.json`
- `hdfs://master11:9000/drugtarget/data/raw/gdc/metadata/cases_samples/run_date=*/cases_samples.tsv`

Output:
- `gdc_file_sample_map`
- `gdc_counts_clean`
- `gdc_counts_clean_protein_coding`
- `quality_check`


In [1]:
from datetime import datetime
from functools import reduce
import glob
import json
import os
import re
import socket
import sys
import time
from urllib import request as urlrequest

# Kernel Jupyter đôi khi chưa có PySpark trong PYTHONPATH.
# Cell này tự thêm thư viện Spark/Py4J local để notebook chạy được trực tiếp.
SPARK_HOME = os.environ.get("SPARK_HOME", "/home/dis/spark-3.5.1-bin-hadoop3")
os.environ.setdefault("SPARK_HOME", SPARK_HOME)
spark_python_dir = os.path.join(SPARK_HOME, "python")
if os.path.isdir(spark_python_dir) and spark_python_dir not in sys.path:
    sys.path.insert(0, spark_python_dir)
for py4j_zip in glob.glob(os.path.join(SPARK_HOME, "python", "lib", "py4j-*.zip")):
    if py4j_zip not in sys.path:
        sys.path.insert(0, py4j_zip)

from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F, types as T


# Cấu hình Spark cluster hiện tại. Notebook ưu tiên worker để chạy song song,
# nhưng nếu không phát hiện worker ALIVE thì sẽ cảnh báo và chuyển về local[*].
SPARK_MASTER_HOST = "master11"
SPARK_MASTER_PORT = 7077
SPARK_CLUSTER_MASTER = f"spark://{SPARK_MASTER_HOST}:{SPARK_MASTER_PORT}"
SPARK_MASTER_WEB_UI = f"http://{SPARK_MASTER_HOST}:8080"
SPARK_DRIVER_HOST = "master11"

# Các tham số an toàn cho workload count TSV: nhiều file nhỏ, tổng dữ liệu vừa phải.
# Mục tiêu là chia việc đủ đều cho worker nhưng không tạo quá nhiều task/shuffle nhỏ.
SAFE_DRIVER_MEMORY = "4g"
SAFE_EXECUTOR_MEMORY = "3g"
SAFE_EXECUTOR_CORES = 1
SAFE_MAX_CORES_PER_WORKER = 2
SAFE_MIN_SHUFFLE_PARTITIONS = 16
SAFE_MAX_SHUFFLE_PARTITIONS = 96
SAFE_MAX_PARTITION_BYTES = "32m"
SAFE_OPEN_COST_BYTES = "8m"


def _is_spark_master_reachable(host: str, port: int, timeout_seconds: int = 2) -> bool:
    """Kiểm tra Spark master có mở cổng nhận application hay không."""
    try:
        with socket.create_connection((host, port), timeout=timeout_seconds):
            return True
    except OSError:
        return False


def _get_spark_master_status(timeout_seconds: int = 2) -> dict:
    """Đọc JSON từ Spark Master UI để biết cluster có worker sống hay không."""
    try:
        with urlrequest.urlopen(f"{SPARK_MASTER_WEB_UI}/json/", timeout=timeout_seconds) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        print(f"Không đọc được Spark Master UI JSON, sẽ cân nhắc fallback local: {exc}")
        return {}


def _get_alive_workers(master_status: dict) -> list:
    """Lấy danh sách worker ALIVE và còn core để chạy executor."""
    workers = master_status.get("workers") or []
    alive_workers = []
    for worker in workers:
        worker_state = str(worker.get("state", "ALIVE")).upper()
        worker_cores = int(worker.get("cores", worker.get("coresfree", 0)) or 0)
        if worker_state == "ALIVE" and worker_cores > 0:
            alive_workers.append(worker)
    return alive_workers


def _stop_existing_spark_session() -> None:
    """Dừng SparkSession cũ trong kernel để tránh reuse sai master khi chạy lại notebook."""
    existing_spark = globals().get("spark")
    if existing_spark is None:
        return
    try:
        existing_spark.stop()
        time.sleep(2)
    except Exception as exc:
        print(f"Không thể stop SparkSession cũ, tiếp tục tạo session mới: {exc}")


def _get_executor_count(spark_session) -> int:
    """Đếm executor của app hiện tại, không tính driver."""
    try:
        memory_status_size = spark_session.sparkContext._jsc.sc().getExecutorMemoryStatus().size()
        return max(memory_status_size - 1, 0)
    except Exception:
        return 0


def _wait_for_executors(spark_session, timeout_seconds: int = 20) -> int:
    """Đợi worker cấp executor trước khi kết luận app cluster có chạy song song được hay không."""
    deadline = time.time() + timeout_seconds
    executor_count = _get_executor_count(spark_session)
    while time.time() < deadline and executor_count == 0:
        time.sleep(2)
        executor_count = _get_executor_count(spark_session)
    return executor_count


def _safe_shuffle_partitions(alive_worker_count: int) -> int:
    """Tính số shuffle partition vừa đủ theo số worker, có chặn min/max để tránh nghẽn."""
    if alive_worker_count <= 0:
        return SAFE_MIN_SHUFFLE_PARTITIONS
    return min(max(alive_worker_count * 24, SAFE_MIN_SHUFFLE_PARTITIONS), SAFE_MAX_SHUFFLE_PARTITIONS)


def _build_spark_session(master_url: str, shuffle_partitions: int, use_cluster: bool):
    """Tạo SparkSession với cấu hình I/O và shuffle phù hợp cho pipeline này."""
    builder = (
        SparkSession.builder.appName("drugtarget-gdc-cleaning")
        .master(master_url)
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.parquet.compression.codec", "snappy")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        .config("spark.sql.files.maxPartitionBytes", SAFE_MAX_PARTITION_BYTES)
        .config("spark.sql.files.openCostInBytes", SAFE_OPEN_COST_BYTES)
        .config("spark.sql.shuffle.partitions", str(shuffle_partitions))
        .config("spark.default.parallelism", str(shuffle_partitions))
        .config("spark.hadoop.fs.defaultFS", "hdfs://master11:9000")
        .config("spark.driver.memory", SAFE_DRIVER_MEMORY)
    )

    if use_cluster:
        builder = (
            builder.config("spark.driver.host", SPARK_DRIVER_HOST)
            .config("spark.executor.memory", SAFE_EXECUTOR_MEMORY)
            .config("spark.executor.cores", str(SAFE_EXECUTOR_CORES))
            .config("spark.cores.max", str(max(SAFE_MAX_CORES_PER_WORKER, shuffle_partitions // 12)))
        )

    spark_session = builder.getOrCreate()
    spark_session.sparkContext.setLogLevel("WARN")
    return spark_session


# Kiểm tra worker trước khi tạo SparkSession. Nếu không có worker ALIVE thì bắt buộc fallback local.
_stop_existing_spark_session()
master_status = _get_spark_master_status()
alive_workers = _get_alive_workers(master_status)
alive_worker_count = len(alive_workers)
master_reachable = _is_spark_master_reachable(SPARK_MASTER_HOST, SPARK_MASTER_PORT)
use_cluster = master_reachable and alive_worker_count > 0

if not use_cluster:
    print("WARN: Không phát hiện Spark worker ALIVE, chuyển sang local[*].")

shuffle_partitions = _safe_shuffle_partitions(alive_worker_count if use_cluster else 0)
spark = _build_spark_session(
    SPARK_CLUSTER_MASTER if use_cluster else "local[*]",
    shuffle_partitions,
    use_cluster,
)

# Nếu Spark master có worker nhưng app không nhận executor, notebook vẫn fallback local để tránh chạy treo.
if use_cluster:
    executor_count = _wait_for_executors(spark, timeout_seconds=20)
    if executor_count == 0:
        print("WARN: Spark master có worker nhưng app không nhận executor, chuyển sang local[*].")
        spark.stop()
        time.sleep(2)
        use_cluster = False
        shuffle_partitions = _safe_shuffle_partitions(0)
        spark = _build_spark_session("local[*]", shuffle_partitions, use_cluster=False)

print(f"Spark master đang dùng: {spark.sparkContext.master}")
print(f"Số worker ALIVE phát hiện từ master UI: {alive_worker_count}")
print(f"Số executor hiện có, không tính driver: {_get_executor_count(spark)}")
print(f"spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/24 15:21:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master đang dùng: spark://master11:7077
Số worker ALIVE phát hiện từ master UI: 2
Số executor hiện có, không tính driver: 1


spark.sql.shuffle.partitions = 48


In [2]:
# Khai báo đường dẫn HDFS và tên bảng output.
# Dùng HDFS URI đầy đủ để notebook không phụ thuộc vào defaultFS của môi trường shell.
HDFS_BASE = "hdfs://master11:9000"
GDC_RAW_BASE = f"{HDFS_BASE}/drugtarget/data/raw/gdc"
GDC_REFINED_BASE = f"{HDFS_BASE}/drugtarget/data/refined/gdc"

COUNTS_PATH = f"{GDC_RAW_BASE}/counts/file_id=*/*.tsv"
FILES_INDEX_PATH = f"{GDC_RAW_BASE}/metadata/files_index/run_date=*/files_index_from_*.json"
CASES_SAMPLES_PATH = f"{GDC_RAW_BASE}/metadata/cases_samples/run_date=*/cases_samples.tsv"

OUTPUT_PATHS = {
    "gdc_file_sample_map": f"{GDC_REFINED_BASE}/gdc_file_sample_map",
    "gdc_counts_clean": f"{GDC_REFINED_BASE}/gdc_counts_clean",
    "gdc_counts_clean_protein_coding": f"{GDC_REFINED_BASE}/gdc_counts_clean_protein_coding",
    "quality_check": f"{GDC_REFINED_BASE}/quality_check",
}


def normalize_sample_group(sample_type_col, tissue_type_col=None):
    """Chuẩn hóa sample type GDC về 3 nhóm chính: tumor, normal, other."""
    sample_text = F.lower(F.coalesce(sample_type_col.cast("string"), F.lit("")))
    tissue_text = F.lower(F.coalesce(tissue_type_col.cast("string"), F.lit(""))) if tissue_type_col is not None else F.lit("")
    combined_text = F.concat_ws(" ", sample_text, tissue_text)
    return (
        F.when(combined_text.rlike("normal"), F.lit("normal"))
        .when(combined_text.rlike("tumou?r|tumor|recurrent|metastatic|primary"), F.lit("tumor"))
        .otherwise(F.lit("other"))
    )


def write_parquet_overwrite(df, path: str, partition_cols=None) -> None:
    """Ghi đè DataFrame ra Parquet Snappy, có partition nếu bảng lớn cần chia theo sample_group."""
    writer = df.write.mode("overwrite").option("compression", "snappy")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(path)
    print(f"Đã ghi overwrite: {path}")


def assert_zero(value: int, message: str) -> None:
    """Helper assert để thông báo rõ số bản ghi lỗi khi kiểm tra chất lượng."""
    if value != 0:
        raise AssertionError(f"{message}. Số bản ghi lỗi: {value}")


In [3]:
# Đọc files_index từ GDC.
# Đây là nguồn ưu tiên vì nó nối trực tiếp file count với case/sample tương ứng.
files_index_raw = spark.read.option("multiLine", "true").json(FILES_INDEX_PATH)

files_index_hits = files_index_raw.select(F.explode_outer("data.hits").alias("hit"))

files_index_sample_map = (
    files_index_hits
    .select(
        F.col("hit.file_id").alias("file_id"),
        F.col("hit.file_name").alias("file_name"),
        F.explode_outer("hit.cases").alias("case"),
    )
    .select(
        "file_id",
        "file_name",
        F.col("case.case_id").alias("case_id"),
        F.explode_outer("case.samples").alias("sample"),
    )
    .select(
        "file_id",
        "file_name",
        "case_id",
        F.col("sample.sample_id").alias("sample_id"),
        F.col("sample.sample_type").alias("sample_type"),
        F.col("sample.tissue_type").alias("tissue_type"),
    )
    .filter(F.col("file_id").isNotNull())
    .withColumn("sample_group", normalize_sample_group(F.col("sample_type"), F.col("tissue_type")))
    .dropDuplicates(["file_id", "case_id", "sample_id"])
)

print("Schema files_index_sample_map:")
files_index_sample_map.printSchema()
print(f"Số dòng map từ files_index: {files_index_sample_map.count()}")


Schema files_index_sample_map:
root
 |-- file_id: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- tissue_type: string (nullable = true)
 |-- sample_group: string (nullable = false)



Số dòng map từ files_index: 601


In [4]:
# Đọc cases_samples.tsv để bổ sung/đối chiếu metadata sample.
# File này là dạng rộng: samples.0.*, samples.1.*, ... nên cần unpivot thủ công theo index sample.
cases_samples_raw = (
    spark.read.option("header", "true")
    .option("sep", "\t")
    .option("multiLine", "false")
    .csv(CASES_SAMPLES_PATH)
)

sample_indices = sorted(
    {
        int(match.group(1))
        for column_name in cases_samples_raw.columns
        for match in [re.match(r"samples\.(\d+)\.sample_id$", column_name)]
        if match
    }
)

case_sample_frames = []
for sample_index in sample_indices:
    sample_id_col = f"samples.{sample_index}.sample_id"
    sample_type_col = f"samples.{sample_index}.sample_type"
    tissue_type_col = f"samples.{sample_index}.tissue_type"

    # Một số export có thể thiếu vài cột ở sample index cao, vì vậy chỉ lấy khi cột tồn tại.
    if sample_id_col not in cases_samples_raw.columns:
        continue

    frame = cases_samples_raw.select(
        F.col("case_id").alias("case_id"),
        F.col(f"`{sample_id_col}`").alias("sample_id"),
        (
            F.col(f"`{sample_type_col}`").alias("sample_type")
            if sample_type_col in cases_samples_raw.columns
            else F.lit(None).cast("string").alias("sample_type")
        ),
        (
            F.col(f"`{tissue_type_col}`").alias("tissue_type")
            if tissue_type_col in cases_samples_raw.columns
            else F.lit(None).cast("string").alias("tissue_type")
        ),
    ).filter(F.col("sample_id").isNotNull() & (F.length(F.trim(F.col("sample_id"))) > 0))
    case_sample_frames.append(frame)

case_samples_schema = T.StructType(
    [
        T.StructField("case_id", T.StringType(), True),
        T.StructField("sample_id", T.StringType(), True),
        T.StructField("sample_type", T.StringType(), True),
        T.StructField("tissue_type", T.StringType(), True),
    ]
)

if case_sample_frames:
    cases_samples_dim = reduce(lambda left, right: left.unionByName(right), case_sample_frames)
else:
    cases_samples_dim = spark.createDataFrame([], case_samples_schema)

cases_samples_dim = (
    cases_samples_dim
    .withColumn("sample_group", normalize_sample_group(F.col("sample_type"), F.col("tissue_type")))
    .dropDuplicates(["case_id", "sample_id"])
)

print("Schema cases_samples_dim:")
cases_samples_dim.printSchema()
print(f"Số sample metadata từ cases_samples.tsv: {cases_samples_dim.count()}")


Schema cases_samples_dim:
root
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- tissue_type: string (nullable = true)
 |-- sample_group: string (nullable = false)



Số sample metadata từ cases_samples.tsv: 1644


In [5]:
# Đọc toàn bộ STAR count TSV từ HDFS.
# option comment="#" giúp Spark bỏ dòng "# gene-model: GENCODE v36" trước khi nhận header.
# Các dòng kỹ thuật N_unmapped/N_multimapping/... vẫn được đọc nhưng sẽ bị lọc khỏi bảng gene.
counts_raw = (
    spark.read.option("header", "true")
    .option("sep", "\t")
    .option("comment", "#")
    .option("mode", "PERMISSIVE")
    .csv(COUNTS_PATH)
    .withColumn("_count_file_path", F.input_file_name())
    .withColumn("file_id", F.regexp_extract(F.col("_count_file_path"), r"file_id=([^/]+)", 1))
    .withColumn("_count_file_name", F.element_at(F.split(F.col("_count_file_path"), "/"), -1))
)

# Inventory count file chỉ dùng nội bộ để đánh dấu has_count_file và join count với metadata.
# Không ghi _count_file_path ra bảng gdc_file_sample_map theo yêu cầu.
count_file_inventory = (
    counts_raw
    .select("file_id", "_count_file_name", "_count_file_path")
    .filter(F.col("file_id") != "")
    .dropDuplicates(["file_id", "_count_file_path"])
)

count_file_inventory_by_file = (
    count_file_inventory
    .groupBy("file_id")
    .agg(
        F.first("_count_file_name", ignorenulls=True).alias("_count_file_name"),
        F.first("_count_file_path", ignorenulls=True).alias("_count_file_path"),
    )
)

print("Schema count_file_inventory_by_file:")
count_file_inventory_by_file.printSchema()
print(f"Số file count tìm thấy trên HDFS: {count_file_inventory_by_file.count()}")


Schema count_file_inventory_by_file:
root
 |-- file_id: string (nullable = false)
 |-- _count_file_name: string (nullable = true)
 |-- _count_file_path: string (nullable = true)



Số file count tìm thấy trên HDFS: 601


In [6]:
# Tạo bảng gdc_file_sample_map.
# Ưu tiên metadata từ files_index, bổ sung sample_type/sample_group từ cases_samples nếu files_index thiếu.
metadata_enriched = (
    files_index_sample_map.alias("idx")
    .join(
        cases_samples_dim.select(
            "case_id",
            "sample_id",
            F.col("sample_type").alias("sample_type_from_cases"),
            F.col("sample_group").alias("sample_group_from_cases"),
        ).alias("cs"),
        on=["case_id", "sample_id"],
        how="left",
    )
    .select(
        F.col("idx.file_id"),
        F.col("idx.file_name"),
        F.col("idx.case_id"),
        F.col("idx.sample_id"),
        F.coalesce(F.col("idx.sample_type"), F.col("cs.sample_type_from_cases")).alias("sample_type"),
        F.coalesce(F.col("idx.sample_group"), F.col("cs.sample_group_from_cases"), F.lit("other")).alias("sample_group"),
    )
)

file_sample_map_internal_raw = (
    metadata_enriched.alias("m")
    .join(count_file_inventory_by_file.alias("c"), on="file_id", how="full_outer")
    .select(
        F.coalesce(F.col("m.file_id"), F.col("c.file_id")).alias("file_id"),
        F.coalesce(F.col("m.file_name"), F.col("c._count_file_name")).alias("file_name"),
        F.col("m.case_id").alias("case_id"),
        F.col("m.sample_id").alias("sample_id"),
        F.col("m.sample_type").alias("sample_type"),
        F.coalesce(F.col("m.sample_group"), F.lit("other")).alias("sample_group"),
        F.col("c._count_file_path").isNotNull().alias("has_count_file"),
        F.col("c._count_file_path").alias("_count_file_path"),
    )
    .filter(F.col("file_id").isNotNull() & (F.length(F.trim(F.col("file_id"))) > 0))
)

duplicate_file_rows = (
    file_sample_map_internal_raw
    .groupBy("file_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
if duplicate_file_rows > 0:
    print(f"WARN: Có {duplicate_file_rows} file_id xuất hiện nhiều hơn 1 dòng, sẽ gom về 1 dòng/file_id.")

# Gom về một dòng cho mỗi file_id để bảng map có khóa file_id ổn định.
file_sample_map_internal = (
    file_sample_map_internal_raw
    .groupBy("file_id")
    .agg(
        F.first("file_name", ignorenulls=True).alias("file_name"),
        F.first("case_id", ignorenulls=True).alias("case_id"),
        F.first("sample_id", ignorenulls=True).alias("sample_id"),
        F.first("sample_type", ignorenulls=True).alias("sample_type"),
        F.first("sample_group", ignorenulls=True).alias("sample_group"),
        F.max(F.col("has_count_file").cast("int")).cast("boolean").alias("has_count_file"),
        F.first("_count_file_path", ignorenulls=True).alias("_count_file_path"),
    )
)

gdc_file_sample_map = file_sample_map_internal.select(
    "file_id",
    "file_name",
    "case_id",
    "sample_id",
    "sample_type",
    "sample_group",
    "has_count_file",
)

print("Schema gdc_file_sample_map:")
gdc_file_sample_map.printSchema()
print(f"Số dòng gdc_file_sample_map: {gdc_file_sample_map.count()}")


Schema gdc_file_sample_map:
root
 |-- file_id: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- has_count_file: boolean (nullable = true)



Số dòng gdc_file_sample_map: 601


In [7]:
# Tạo bảng gdc_counts_clean.
# Chỉ giữ các dòng gene ENSG; các dòng N_* là metric kỹ thuật của STAR và không phải gene.
counts_gene_rows = (
    counts_raw
    .filter(F.col("gene_id").rlike("^ENSG"))
    .withColumn("is_par_y", F.col("gene_id").endswith("_PAR_Y"))
    .withColumn("_gene_id_without_par_y", F.regexp_replace(F.col("gene_id"), "_PAR_Y$", ""))
    .withColumn("gene_id_base", F.regexp_replace(F.col("_gene_id_without_par_y"), r"\.[0-9]+$", ""))
    .withColumn("raw_count", F.col("unstranded").cast("long"))
    .withColumn("tpm", F.col("tpm_unstranded").cast("double"))
    .withColumn("fpkm", F.col("fpkm_unstranded").cast("double"))
    .withColumn("fpkm_uq", F.col("fpkm_uq_unstranded").cast("double"))
)

gdc_counts_clean = (
    counts_gene_rows.alias("cnt")
    .join(
        gdc_file_sample_map.select(
            "file_id",
            "case_id",
            "sample_id",
            "sample_type",
            "sample_group",
        ).alias("map"),
        on="file_id",
        how="left",
    )
    .select(
        "file_id",
        "case_id",
        "sample_id",
        "sample_type",
        "sample_group",
        F.col("cnt.gene_id").alias("gene_id"),
        "gene_id_base",
        "is_par_y",
        F.col("cnt.gene_name").alias("gene_name"),
        F.col("cnt.gene_type").alias("gene_type"),
        "raw_count",
        "tpm",
        "fpkm",
        "fpkm_uq",
    )
)

# Persist vì bảng clean được dùng lại cho protein-coding, QC và ghi output.
gdc_counts_clean = gdc_counts_clean.persist(StorageLevel.DISK_ONLY)

print("Schema gdc_counts_clean:")
gdc_counts_clean.printSchema()
print(f"Số dòng gdc_counts_clean: {gdc_counts_clean.count()}")


Schema gdc_counts_clean:
root
 |-- file_id: string (nullable = false)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- is_par_y: boolean (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_type: string (nullable = true)
 |-- raw_count: long (nullable = true)
 |-- tpm: double (nullable = true)
 |-- fpkm: double (nullable = true)
 |-- fpkm_uq: double (nullable = true)



Số dòng gdc_counts_clean: 36456660


In [8]:
# Tạo bảng gdc_counts_clean_protein_coding.
# Bảng này chỉ phục vụ phân tích gene protein-coding và loại bỏ gene PAR_Y theo yêu cầu.
gdc_counts_clean_protein_coding = (
    gdc_counts_clean
    .filter((F.col("gene_type") == "protein_coding") & (~F.col("is_par_y")))
    .drop("is_par_y")
)

print("Schema gdc_counts_clean_protein_coding:")
gdc_counts_clean_protein_coding.printSchema()
print(f"Số dòng gdc_counts_clean_protein_coding: {gdc_counts_clean_protein_coding.count()}")


Schema gdc_counts_clean_protein_coding:
root
 |-- file_id: string (nullable = false)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_type: string (nullable = true)
 |-- raw_count: long (nullable = true)
 |-- tpm: double (nullable = true)
 |-- fpkm: double (nullable = true)
 |-- fpkm_uq: double (nullable = true)



Số dòng gdc_counts_clean_protein_coding: 11986344


In [9]:
# Tạo bảng quality_check.
# Mỗi dòng tương ứng với một file/sample; các metric giúp phát hiện sample bất thường.
quality_base = (
    gdc_counts_clean
    .groupBy("file_id", "case_id", "sample_id", "sample_type", "sample_group")
    .agg(
        F.sum(F.coalesce(F.col("raw_count"), F.lit(0))).cast("long").alias("total_raw_count"),
        F.sum(F.when(F.col("raw_count") > 0, 1).otherwise(0)).cast("long").alias("num_detected_genes_raw_gt_0"),
        F.sum(F.when(F.col("tpm") > 1, 1).otherwise(0)).cast("long").alias("num_detected_genes_tpm_gt_1"),
        F.expr("percentile_approx(tpm, 0.5, 10000)").cast("double").alias("median_tpm"),
        F.sum(F.when(F.col("raw_count") == 0, 1).otherwise(0)).cast("long").alias("_num_zero_genes"),
        F.count(F.lit(1)).cast("long").alias("_num_genes"),
        F.sum(F.when(F.col("gene_type") == "protein_coding", 1).otherwise(0)).cast("long").alias("num_protein_coding_genes"),
    )
    .withColumn(
        "pct_zero_genes",
        F.when(F.col("_num_genes") > 0, F.col("_num_zero_genes") / F.col("_num_genes")).otherwise(F.lit(None).cast("double")),
    )
)

qc_percentiles = quality_base.agg(
    F.expr("percentile_approx(total_raw_count, array(0.25, 0.75), 10000)").alias("library_size_q"),
    F.expr("percentile_approx(num_detected_genes_raw_gt_0, array(0.25, 0.75), 10000)").alias("detected_genes_q"),
).collect()[0]


def iqr_bounds(q_values):
    """Tính ngưỡng outlier IQR; nếu IQR bằng 0 thì trả None để không flag sai hàng loạt."""
    if not q_values or len(q_values) != 2 or q_values[0] is None or q_values[1] is None:
        return None
    q1, q3 = float(q_values[0]), float(q_values[1])
    iqr = q3 - q1
    if iqr <= 0:
        return None
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


library_bounds = iqr_bounds(qc_percentiles["library_size_q"])
detected_bounds = iqr_bounds(qc_percentiles["detected_genes_q"])

quality_check = quality_base
if library_bounds is None:
    quality_check = quality_check.withColumn("is_outlier_library_size", F.lit(False))
else:
    library_lower, library_upper = library_bounds
    quality_check = quality_check.withColumn(
        "is_outlier_library_size",
        (F.col("total_raw_count") < F.lit(library_lower)) | (F.col("total_raw_count") > F.lit(library_upper)),
    )

if detected_bounds is None:
    quality_check = quality_check.withColumn("is_outlier_detected_genes", F.lit(False))
else:
    detected_lower, detected_upper = detected_bounds
    quality_check = quality_check.withColumn(
        "is_outlier_detected_genes",
        (F.col("num_detected_genes_raw_gt_0") < F.lit(detected_lower))
        | (F.col("num_detected_genes_raw_gt_0") > F.lit(detected_upper)),
    )

quality_check = quality_check.select(
    "file_id",
    "case_id",
    "sample_id",
    "sample_type",
    "sample_group",
    "total_raw_count",
    "num_detected_genes_raw_gt_0",
    "num_detected_genes_tpm_gt_1",
    "median_tpm",
    "pct_zero_genes",
    "num_protein_coding_genes",
    "is_outlier_library_size",
    "is_outlier_detected_genes",
)

print("Ngưỡng IQR library size:", library_bounds)
print("Ngưỡng IQR detected genes:", detected_bounds)
print("Schema quality_check:")
quality_check.printSchema()
print(f"Số dòng quality_check: {quality_check.count()}")


Ngưỡng IQR library size: (-10777708.0, 105165948.0)
Ngưỡng IQR detected genes: (25140.0, 40172.0)
Schema quality_check:
root
 |-- file_id: string (nullable = false)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- total_raw_count: long (nullable = true)
 |-- num_detected_genes_raw_gt_0: long (nullable = true)
 |-- num_detected_genes_tpm_gt_1: long (nullable = true)
 |-- median_tpm: double (nullable = true)
 |-- pct_zero_genes: double (nullable = true)
 |-- num_protein_coding_genes: long (nullable = true)
 |-- is_outlier_library_size: boolean (nullable = true)
 |-- is_outlier_detected_genes: boolean (nullable = true)



Số dòng quality_check: 601


In [10]:
# Kiểm tra chất lượng trước khi ghi output.
# Các assert này giúp notebook fail sớm nếu schema/join bị lệch so với kỳ vọng.
duplicate_file_ids = (
    gdc_file_sample_map
    .groupBy("file_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert_zero(duplicate_file_ids, "gdc_file_sample_map có duplicate file_id")

missing_file_map_ids = (
    gdc_counts_clean.select("file_id").distinct()
    .join(gdc_file_sample_map.select("file_id").distinct(), on="file_id", how="left_anti")
    .count()
)
assert_zero(missing_file_map_ids, "Có file_id trong gdc_counts_clean không có trong gdc_file_sample_map")

bad_protein_coding_type = (
    gdc_counts_clean_protein_coding
    .filter(F.col("gene_type") != "protein_coding")
    .count()
)
assert_zero(bad_protein_coding_type, "Bảng protein-coding có gene_type khác protein_coding")

bad_protein_coding_par_y = (
    gdc_counts_clean_protein_coding
    .filter(F.col("gene_id").endswith("_PAR_Y"))
    .count()
)
assert_zero(bad_protein_coding_par_y, "Bảng protein-coding vẫn còn gene PAR_Y")

qc_total_rows = quality_check.count()
qc_distinct_keys = quality_check.select("file_id", "sample_id").distinct().count()
if qc_total_rows != qc_distinct_keys:
    raise AssertionError(
        f"quality_check không có đúng một dòng mỗi file_id/sample_id: total={qc_total_rows}, distinct={qc_distinct_keys}"
    )

bad_pct_zero = (
    quality_check
    .filter(
        F.col("pct_zero_genes").isNull()
        | (F.col("pct_zero_genes") < 0)
        | (F.col("pct_zero_genes") > 1)
    )
    .count()
)
assert_zero(bad_pct_zero, "quality_check có pct_zero_genes ngoài khoảng [0, 1]")

print("Tất cả kiểm tra trước ghi đều đạt.")


Tất cả kiểm tra trước ghi đều đạt.


In [11]:
# Ghi đè 4 bảng refined ra HDFS ở dạng Parquet Snappy.
# Bảng count lớn được partition theo sample_group để đọc lại nhanh theo tumor/normal/other.
write_start = datetime.utcnow()
print(f"Bắt đầu ghi output lúc UTC: {write_start.isoformat()}Z")

write_parquet_overwrite(gdc_file_sample_map, OUTPUT_PATHS["gdc_file_sample_map"])
write_parquet_overwrite(gdc_counts_clean, OUTPUT_PATHS["gdc_counts_clean"], partition_cols=["sample_group"])
write_parquet_overwrite(
    gdc_counts_clean_protein_coding,
    OUTPUT_PATHS["gdc_counts_clean_protein_coding"],
    partition_cols=["sample_group"],
)
write_parquet_overwrite(quality_check, OUTPUT_PATHS["quality_check"])

print(f"Hoàn tất ghi output lúc UTC: {datetime.utcnow().isoformat()}Z")


/tmp/ipykernel_25359/1541505564.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  write_start = datetime.utcnow()


Bắt đầu ghi output lúc UTC: 2026-05-24T08:34:33.435660Z


Đã ghi overwrite: hdfs://master11:9000/drugtarget/data/refined/gdc/gdc_file_sample_map


Đã ghi overwrite: hdfs://master11:9000/drugtarget/data/refined/gdc/gdc_counts_clean


Đã ghi overwrite: hdfs://master11:9000/drugtarget/data/refined/gdc/gdc_counts_clean_protein_coding


Đã ghi overwrite: hdfs://master11:9000/drugtarget/data/refined/gdc/quality_check
Hoàn tất ghi output lúc UTC: 2026-05-24T08:51:24.518694Z


/tmp/ipykernel_25359/1541505564.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"Hoàn tất ghi output lúc UTC: {datetime.utcnow().isoformat()}Z")


In [12]:
# Đọc lại các bảng Parquet vừa ghi để xác nhận output tồn tại, schema đúng và có dữ liệu.
for table_name, table_path in OUTPUT_PATHS.items():
    print("=" * 100)
    print(f"Bảng: {table_name}")
    print(f"Đường dẫn: {table_path}")
    reloaded_df = spark.read.parquet(table_path)
    reloaded_df.printSchema()
    print(f"Row count: {reloaded_df.count()}")

# Giải phóng cache sau khi đã ghi và kiểm tra xong.
gdc_counts_clean.unpersist()
print("Pipeline GDC cleaning hoàn tất.")


Bảng: gdc_file_sample_map
Đường dẫn: hdfs://master11:9000/drugtarget/data/refined/gdc/gdc_file_sample_map
root
 |-- file_id: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- has_count_file: boolean (nullable = true)



Row count: 601
Bảng: gdc_counts_clean
Đường dẫn: hdfs://master11:9000/drugtarget/data/refined/gdc/gdc_counts_clean
root
 |-- file_id: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- is_par_y: boolean (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_type: string (nullable = true)
 |-- raw_count: long (nullable = true)
 |-- tpm: double (nullable = true)
 |-- fpkm: double (nullable = true)
 |-- fpkm_uq: double (nullable = true)
 |-- sample_group: string (nullable = true)



Row count: 36456660
Bảng: gdc_counts_clean_protein_coding
Đường dẫn: hdfs://master11:9000/drugtarget/data/refined/gdc/gdc_counts_clean_protein_coding


root
 |-- file_id: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_type: string (nullable = true)
 |-- raw_count: long (nullable = true)
 |-- tpm: double (nullable = true)
 |-- fpkm: double (nullable = true)
 |-- fpkm_uq: double (nullable = true)
 |-- sample_group: string (nullable = true)



Row count: 11986344
Bảng: quality_check
Đường dẫn: hdfs://master11:9000/drugtarget/data/refined/gdc/quality_check
root
 |-- file_id: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_id: string (nullable = true)
 |-- sample_type: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- total_raw_count: long (nullable = true)
 |-- num_detected_genes_raw_gt_0: long (nullable = true)
 |-- num_detected_genes_tpm_gt_1: long (nullable = true)
 |-- median_tpm: double (nullable = true)
 |-- pct_zero_genes: double (nullable = true)
 |-- num_protein_coding_genes: long (nullable = true)
 |-- is_outlier_library_size: boolean (nullable = true)
 |-- is_outlier_detected_genes: boolean (nullable = true)



Row count: 601
Pipeline GDC cleaning hoàn tất.
